#### Multi-Period Transit Lead Times ($L_{o,d}$) Analysis


### 1. Mathematical Formulation

#### Sets and Parameters
- $\mathcal{O}$: Set of origin nodes $o$.
- $\mathcal{D}$: Set of destination nodes $d$.
- $\mathcal{L} \subseteq \mathcal{O} \times \mathcal{D}$: Set of active shipping lanes $(o,d)$.
- $\mathcal{T} = [t_0, t_1, \dots, t_{N-1}]$: Ordered sequence of $N$ planning periods.
- $L_{o,d} \in \mathbb{N}_0$: Integer transit lead time (in periods) for lane $(o,d)$.
- $C_{o, t_k}$: Daily dispatch capacity at origin $o$ in period $t_k$.
- $D_{d, t_k}$: Demand at destination $d$ in period $t_k$.
- $c_{o,d}$: Unit transportation cost on lane $(o,d)$.
- $h_d$: Unit inventory holding cost per period at destination $d$.

#### Decision Variables
- $x_{o,d,t_k} \ge 0$: Flow dispatched from origin $o$ to destination $d$ at period $t_k$.
- $v_{d,t_k} \ge 0$: On-hand inventory level at destination $d$ at the end of period $t_k$.

#### Baseline Model ($L_{o,d} = 0$)
Shipments arrive instantaneously in the period they are dispatched:
$$\text{inflow}(d, t_k) = \sum_{o: (o,d) \in \mathcal{L}} x_{o,d,t_k}$$
$$v_{d,t_k} = v_{d,t_{k-1}} + \text{inflow}(d, t_k) - D_{d,t_k} \quad \forall d \in \mathcal{D}, \, k \in \{0, \dots, N-1\}$$
$$\sum_{d: (o,d) \in \mathcal{L}} x_{o,d,t_k} \le C_{o,t_k} \quad \forall o \in \mathcal{O}, \, k \in \{0, \dots, N-1\}$$

#### Lead Time Model ($L_{o,d} \ge 0$)
Shipments dispatched at $t_{k - L_{o,d}}$ arrive at period $t_k$:
$$\text{inflow}(d, t_k) = \sum_{\substack{o: (o,d) \in \mathcal{L} \\ k - L_{o,d} \ge 0}} x_{o,d, t_{k - L_{o,d}}}$$
$$v_{d,t_k} = v_{d,t_{k-1}} + \text{inflow}(d, t_k) - D_{d,t_k} \quad \forall d \in \mathcal{D}, \, k \in \{0, \dots, N-1\}$$

Origin capacity $C_{o,t_k}$ is evaluated at the dispatch period $t_k$:
$$\sum_{d: (o,d) \in \mathcal{L}} x_{o,d,t_k} \le C_{o,t_k} \quad \forall o \in \mathcal{O}, \, k \in \{0, \dots, N-1\}$$

In [1]:
import sys
from datetime import date, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

# Path setup
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from optimization.optimizer import MultiPeriodOptimizer

/home/ale/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### 2. Experiment

Test a 4-day horizon ($t_0$ to $t_3$) with a demand spike of 50 units at $t_2$ (`2026-06-03`).
- Scenario A: Baseline ($L = 0$).
- Scenario B: Lead time ($L = 2$ days).

In [2]:
planning_horizon = [date(2026, 6, 1) + timedelta(days=i) for i in range(4)]
dates_str = [d.strftime("%Y-%m-%d") for d in planning_horizon]

demand_ts = pl.DataFrame({
    "destination_id": ["D1"] * 4,
    "date": planning_horizon,
    "demand": [0.0, 0.0, 50.0, 0.0]
})

origins_df = pl.DataFrame({
    "origin_id": ["O1"],
    "daily_capacity": [100.0]
})

destinations_df = pl.DataFrame({
    "destination_id": ["D1"],
    "holding_cost": [1.0]
})

lanes_baseline = pl.DataFrame({
    "origin_id": ["O1"],
    "destination_id": ["D1"],
    "unit_cost": [2.0],
    "lead_time_days": [0]
})

lanes_lead_time = pl.DataFrame({
    "origin_id": ["O1"],
    "destination_id": ["D1"],
    "unit_cost": [2.0],
    "lead_time_days": [2]
})

In [3]:
optimizer = MultiPeriodOptimizer()

result_baseline = optimizer.solve(
    demand_ts=demand_ts,
    origins_df=origins_df,
    lanes_df=lanes_baseline,
    destinations_df=destinations_df,
    planning_horizon=planning_horizon,
    initial_inventory={"D1": 0.0}
)

result_lead_time = optimizer.solve(
    demand_ts=demand_ts,
    origins_df=origins_df,
    lanes_df=lanes_lead_time,
    destinations_df=destinations_df,
    planning_horizon=planning_horizon,
    initial_inventory={"D1": 0.0}
)

print(f"Baseline (L=0) total cost: ${result_baseline.total_cost:.2f}")
print(f"Lead Time (L=2) total cost: ${result_lead_time.total_cost:.2f}")

Baseline (L=0) total cost: $100.00
Lead Time (L=2) total cost: $100.00


In [4]:
def build_timeline_df(result, lead_time_days):
    flow_map = dict(zip(
        [r["period"].strftime("%Y-%m-%d") for r in result.flows.to_dicts()],
        [r["flow"] for r in result.flows.to_dicts()]
    ))
    inv_map = dict(zip(
        [r["period"].strftime("%Y-%m-%d") for r in result.inventory.to_dicts()],
        [r["inventory"] for r in result.inventory.to_dicts()]
    ))
    
    timeline = []
    for idx, p in enumerate(planning_horizon):
        p_str = p.strftime("%Y-%m-%d")
        dispatch = flow_map.get(p_str, 0.0)
        
        arrival_idx = idx + lead_time_days
        arrival_p_str = planning_horizon[arrival_idx].strftime("%Y-%m-%d") if arrival_idx < len(planning_horizon) else "Post-Horizon"
        
        inflow_idx = idx - lead_time_days
        inflow = flow_map.get(planning_horizon[inflow_idx].strftime("%Y-%m-%d"), 0.0) if inflow_idx >= 0 else 0.0
        
        demand_val = demand_ts.filter(pl.col("date") == p)["demand"][0]
        inv_val = inv_map.get(p_str, 0.0)
        
        timeline.append({
            "Period": f"t_{idx} ({p_str})",
            "Dispatched Flow": dispatch,
            "Arrival Period": arrival_p_str if dispatch > 0 else "-",
            "Arrived Inflow": inflow,
            "Demand": demand_val,
            "End Inventory": inv_val
        })
    return pd.DataFrame(timeline)

df_a = build_timeline_df(result_baseline, 0)
df_b = build_timeline_df(result_lead_time, 2)

print("Baseline (L = 0):")
display(df_a)

print("\nLead Time (L = 2):")
display(df_b)

Baseline (L = 0):


,Period,Dispatched Flow,Arrival Period,Arrived Inflow,Demand,End Inventory
0,t_0 (2026-06-01),0.0,-,0.0,0.0,0.0
1,t_1 (2026-06-02),0.0,-,0.0,0.0,0.0
2,t_2 (2026-06-03),50.0,2026-06-03,50.0,50.0,0.0
3,t_3 (2026-06-04),0.0,-,0.0,0.0,0.0



Lead Time (L = 2):


,Period,Dispatched Flow,Arrival Period,Arrived Inflow,Demand,End Inventory
0,t_0 (2026-06-01),50.0,2026-06-03,0.0,0.0,0.0
1,t_1 (2026-06-02),0.0,-,0.0,0.0,0.0
2,t_2 (2026-06-03),0.0,-,50.0,50.0,0.0
3,t_3 (2026-06-04),0.0,-,0.0,0.0,0.0


### Observations
- When $L=2$, the optimizer dispatches flow 2 periods ahead ($t_0$) so that it arrives at $t_2$ to cover demand.
- On-hand destination inventory remains 0 until goods arrive, incurring no holding cost during transit.
- Origin capacity constraints apply at dispatch ($t_0$).
- Setting $L=0$ recovers the baseline instantaneous delivery behavior.